# Assignment 1 — Starter Notebook

**Earnings-Call Sentiment, Event Extraction, and Return Prediction**

This notebook is scaffolding only. It gives you:

1. A transcript parser that splits each `.txt` file into header / prepared-remarks / Q&A pairs / speakers.
2. A price loader (`yfinance`) that computes forward returns at multiple horizons.
3. An **LLM extraction stub** with on-disk caching (Ollama by default; swap in HuggingFace or API).
4. A features DataFrame + train/test split on the first 5–6 calls per ticker.
5. A tiny baseline model and a toy backtest you can replace with something smarter.

**You are encouraged to rip any of this out and replace it.** The goal is a working pipeline, not a sacred structure.

## 0. Environment

Expected packages: `pandas numpy yfinance tqdm requests`. Optional: `transformers torch`, `ollama` client, `anthropic`, `openai`.

If Ollama is your backend, make sure the daemon is running (`ollama serve`) and the model is pulled:
```
ollama pull gemma3:4b   # or qwen3:14b, llama3.1:8b, etc.
```

In [ ]:
# Uncomment if needed in a fresh Colab / venv
# !pip install -q pandas numpy yfinance tqdm requests

import os, re, json, zipfile, pickle, hashlib, time
from pathlib import Path
from datetime import datetime, timedelta
from dataclasses import dataclass, asdict, field
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

## 1. Paths and constants

Adjust `ECT_ZIP` to wherever you saved the transcripts zip.

In [ ]:
ROOT        = Path.cwd()
ECT_ZIP     = Path.home() / 'Documents' / 'Sample Files' / 'ECT.zip'  # <- edit me
TRANSCRIPTS = ROOT / 'transcripts'
CACHE       = ROOT / 'cache'
EXTRACTIONS = CACHE / 'extractions'
PRICES      = CACHE / 'prices'

for d in (TRANSCRIPTS, CACHE, EXTRACTIONS, PRICES):
    d.mkdir(parents=True, exist_ok=True)

# Unzip once
if not any(TRANSCRIPTS.glob('*.txt')):
    with zipfile.ZipFile(ECT_ZIP) as z:
        for name in z.namelist():
            if name.endswith('.txt') and '__MACOSX' not in name:
                with z.open(name) as src, (TRANSCRIPTS / Path(name).name).open('wb') as dst:
                    dst.write(src.read())

txts = sorted(TRANSCRIPTS.glob('*.txt'))
print(f'{len(txts)} transcripts unpacked')
print('Tickers:', sorted({p.name.split("_")[0] for p in txts}))

## 2. Transcript parser

Splits a transcript into:

- `header` — company / quarter / date string on line 1
- `prepared_remarks` — list of `(role, text)` blocks before the Q&A
- `qa_pairs` — list of `{question, q_analyst, answer, a_speaker}` dicts

The S&P format is consistent, but a few transcripts have the `Answer`-without-`Question` glitch the professor
mentioned in class — the parser below tolerates it.

In [ ]:
SECTION_HEADERS = {
    'Presentation Operator Message',
    'Presenter Speech',
    'Question and Answer Operator Message',
    'Question',
    'Answer',
}

ROLE_LINE_RE = re.compile(r'^(Executives|Analysts|Operator)\b.*$')
HEADER_RE    = re.compile(r'^(?P<company>.+?),\s*Q(?P<q>\d)\s*(?P<y>\d{4}).*?Earnings Call.*?(?P<date>[A-Z][a-z]+ \d{1,2},\s*\d{4})')

@dataclass
class Transcript:
    ticker: str
    quarter: str          # e.g. 'Q4-2025'
    call_date: Optional[str]  # YYYY-MM-DD
    company: str
    prepared: List[Dict]  # [{'role': ..., 'text': ...}]
    qa: List[Dict]        # [{'q_role':..., 'question':..., 'a_role':..., 'answer':...}]
    raw_path: str

def _filename_meta(path: Path) -> Tuple[str, str]:
    stem = path.stem   # e.g. AMD_Q4-2025
    ticker, _, q = stem.partition('_')
    return ticker, q

def _blocks(text: str):
    """Yield (section_header, role_line, body_text) tuples."""
    lines = text.splitlines()
    i, n = 0, len(lines)
    while i < n:
        line = lines[i].strip()
        if line in SECTION_HEADERS:
            section = line
            i += 1
            role = lines[i].strip() if i < n else ''
            i += 1
            buf = []
            while i < n and lines[i].strip() not in SECTION_HEADERS:
                buf.append(lines[i])
                i += 1
            yield section, role, '\n'.join(buf).strip()
        else:
            i += 1

def parse_transcript(path: Path) -> Transcript:
    text = path.read_text(errors='ignore')
    first = text.splitlines()[0]
    m = HEADER_RE.match(first)
    if m:
        company = m.group('company').strip()
        date = datetime.strptime(m.group('date').replace('  ', ' '), '%b %d, %Y').strftime('%Y-%m-%d')
    else:
        company, date = '', None

    prepared, qa = [], []
    current_q = None
    in_qa = False

    for section, role, body in _blocks(text):
        if section == 'Question and Answer Operator Message':
            in_qa = True
            continue
        if not in_qa and section == 'Presenter Speech':
            prepared.append({'role': role, 'text': body})
        elif in_qa and section == 'Question':
            current_q = {'q_role': role, 'question': body, 'a_role': None, 'answer': None}
            qa.append(current_q)
        elif in_qa and section == 'Answer':
            if current_q is None or current_q['answer'] is not None:
                # Orphan answer — create a stub
                current_q = {'q_role': None, 'question': '', 'a_role': role, 'answer': body}
                qa.append(current_q)
            else:
                current_q['a_role'] = role
                current_q['answer'] = body
        # 'Presentation Operator Message' / 'Question and Answer Operator Message' bodies are mostly noise — skip

    ticker, quarter = _filename_meta(path)
    return Transcript(
        ticker=ticker, quarter=quarter, call_date=date,
        company=company, prepared=prepared, qa=qa, raw_path=str(path),
    )

# Parse all
transcripts = [parse_transcript(p) for p in tqdm(txts, desc='parsing')]
print(f'Parsed {len(transcripts)} transcripts')

# Sanity check
ex = transcripts[0]
print(f'{ex.ticker} {ex.quarter} on {ex.call_date} — {len(ex.prepared)} prepared blocks, {len(ex.qa)} Q&A pairs')

## 3. Prices & forward returns (via yfinance)

For each call we compute the close-to-close excess return vs. SPY at multiple horizons starting **T+1** to avoid look-ahead.
Cached to disk so you don't hammer Yahoo.

In [ ]:
import yfinance as yf

HORIZONS_DAYS = [1, 5, 21, 63]   # ~1d, 1w, 1m, 3m

def _fetch_prices(ticker: str, start='2023-09-01', end=None) -> pd.DataFrame:
    cache = PRICES / f'{ticker}.parquet'
    if cache.exists() and (time.time() - cache.stat().st_mtime) < 24*3600:
        return pd.read_parquet(cache)
    end = end or datetime.now().strftime('%Y-%m-%d')
    df = yf.Ticker(ticker).history(start=start, end=end, auto_adjust=True)
    df = df[['Close']].reset_index()
    df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None).dt.normalize()
    df.to_parquet(cache)
    time.sleep(0.25)   # be polite
    return df

tickers = sorted({t.ticker for t in transcripts})
prices = {t: _fetch_prices(t) for t in tqdm(tickers + ['SPY'], desc='prices')}

def forward_return(ticker: str, call_date: str, horizon: int, use_excess: bool = True) -> Optional[float]:
    df = prices[ticker]
    spy = prices['SPY']
    d0 = pd.Timestamp(call_date)
    # entry = first trading day strictly after d0 (assume call was after close)
    entry = df[df.Date > d0].head(1)
    if entry.empty: return None
    # exit = entry + horizon trading days
    entry_idx = df.index[df.Date == entry.Date.iloc[0]][0]
    if entry_idx + horizon >= len(df): return None
    r = df.Close.iloc[entry_idx + horizon] / df.Close.iloc[entry_idx] - 1
    if use_excess:
        sp_e = spy[spy.Date == entry.Date.iloc[0]]
        sp_x = spy[spy.Date == df.Date.iloc[entry_idx + horizon]]
        if sp_e.empty or sp_x.empty: return None
        r -= (sp_x.Close.iloc[0] / sp_e.Close.iloc[0] - 1)
    return float(r)

# Attach returns
return_rows = []
for t in transcripts:
    if not t.call_date: continue
    row = {'ticker': t.ticker, 'quarter': t.quarter, 'call_date': t.call_date}
    for h in HORIZONS_DAYS:
        row[f'fwd_excess_{h}d'] = forward_return(t.ticker, t.call_date, h)
    return_rows.append(row)

returns_df = pd.DataFrame(return_rows)
print(returns_df.head())
print(f'NaN rates: {returns_df.isna().mean().round(3).to_dict()}')

## 4. LLM extraction — Ollama backend with on-disk cache

This is the place you'll iterate most. Default backend is Ollama; swap `llm_call` for HuggingFace `transformers`,
Anthropic, or OpenAI as you prefer.

**Gotchas (from class):**
- For Qwen 3 / Gemma 3, pass `"think": false` — otherwise the model burns its token budget on `<think>` blocks.
- Set `num_ctx` high enough for long transcripts (JNJ Q4 2024 is ~185 KB — about 45k tokens).
- Save raw output to disk on parse failure so you can debug the prompt.

In [ ]:
import requests

OLLAMA_HOST = 'http://localhost:11434'
OLLAMA_MODEL = 'gemma3:4b'          # swap for qwen3:14b / llama3.1:8b / mistral-small / ...
OLLAMA_NUM_CTX = 32768

EXTRACT_PROMPT = '''You are a financial analyst. Analyze the following earnings-call transcript and return STRICT JSON with these keys:

{
  "overall_sentiment": <float in [-1,1]>,
  "sentiment_bucket": <"very_bearish"|"bearish"|"neutral"|"bullish"|"very_bullish">,
  "wins":      [<up to 5 short strings, concrete positive events>],
  "risks":     [<up to 5 short strings, concrete negative events>],
  "guidance":  <"raised"|"reaffirmed"|"lowered"|"mixed"|"none">,
  "themes":    [<short thematic tags, e.g. "AI", "china", "pricing", "capex">]
}

Ground every field in the transcript. Do not invent.  Output ONLY the JSON object, no prose.

TRANSCRIPT:
{transcript}
'''

def llm_call(prompt: str, model: str = OLLAMA_MODEL, num_ctx: int = OLLAMA_NUM_CTX, timeout: int = 600) -> str:
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': False,
        'think': False,                           # see CLAUDE.md lesson
        'options': {'think': False, 'num_ctx': num_ctx, 'temperature': 0.1, 'num_predict': 2048},
    }
    r = requests.post(f'{OLLAMA_HOST}/api/generate', json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()['response']

def _salvage_json(raw: str) -> dict:
    """Forgiving parser for open-source LLM JSON output."""
    s = raw
    # strip markdown fences + <think> blocks (closed or unclosed)
    s = re.sub(r'```(?:json)?', '', s)
    s = re.sub(r'<think>.*?(</think>|$)', '', s, flags=re.DOTALL)
    # trim to outermost {...}
    lo, hi = s.find('{'), s.rfind('}')
    if lo >= 0 and hi > lo:
        s = s[lo:hi+1]
    s = re.sub(r',\s*([}\]])', r'\1', s)       # trailing commas
    s = s.replace('True', 'true').replace('False', 'false').replace('None', 'null')
    try:
        return json.loads(s)
    except Exception:
        return {}

def _transcript_text_for_prompt(t: Transcript, max_chars: int = 90_000) -> str:
    """Concatenate prepared + Q&A, truncate defensively."""
    parts = [f'[PREPARED — {b["role"]}]\n{b["text"]}' for b in t.prepared]
    for qa in t.qa:
        parts.append(f'[Q — {qa["q_role"]}] {qa["question"]}\n[A — {qa["a_role"]}] {qa["answer"] or ""}')
    blob = '\n\n'.join(parts)
    return blob[:max_chars]

def extract_one(t: Transcript, force: bool = False) -> dict:
    key = f'{t.ticker}_{t.quarter}_{OLLAMA_MODEL.replace(":","-")}'
    cache = EXTRACTIONS / f'{key}.json'
    raw_cache = EXTRACTIONS / f'{key}.raw.txt'
    if cache.exists() and not force:
        return json.loads(cache.read_text())
    prompt = EXTRACT_PROMPT.format(transcript=_transcript_text_for_prompt(t))
    raw = llm_call(prompt)
    raw_cache.write_text(raw)    # ALWAYS keep the raw
    obj = _salvage_json(raw)
    obj['_ticker'] = t.ticker
    obj['_quarter'] = t.quarter
    obj['_call_date'] = t.call_date
    cache.write_text(json.dumps(obj, indent=2))
    return obj

# Demo on ONE transcript first — iterate on the prompt before scaling up
# demo = extract_one(transcripts[0])
# print(json.dumps(demo, indent=2))

# When you are happy with the prompt, run over all:
# extractions = [extract_one(t) for t in tqdm(transcripts, desc='LLM extract')]

## 5. Feature DataFrame + QoQ deltas

Merge extractions with forward returns, then add change-vs-previous-quarter features.

In [ ]:
def load_extractions() -> pd.DataFrame:
    rows = []
    for p in sorted(EXTRACTIONS.glob('*.json')):
        obj = json.loads(p.read_text())
        rows.append({
            'ticker':   obj.get('_ticker'),
            'quarter':  obj.get('_quarter'),
            'call_date': obj.get('_call_date'),
            'sentiment':  obj.get('overall_sentiment'),
            'sentiment_bucket': obj.get('sentiment_bucket'),
            'n_wins':     len(obj.get('wins', []) or []),
            'n_risks':    len(obj.get('risks', []) or []),
            'guidance':   obj.get('guidance'),
            'themes':     obj.get('themes', []) or [],
            'wins':       obj.get('wins', []) or [],
            'risks':      obj.get('risks', []) or [],
        })
    return pd.DataFrame(rows)

# features = load_extractions()
# features['call_date'] = pd.to_datetime(features['call_date'])
# features = features.sort_values(['ticker','call_date']).reset_index(drop=True)

# # QoQ deltas
# features['sentiment_delta'] = features.groupby('ticker')['sentiment'].diff()
# features['n_risks_delta']   = features.groupby('ticker')['n_risks'].diff()
# features['n_wins_delta']    = features.groupby('ticker')['n_wins'].diff()

# # Risk persistence: fraction of prior-call risks that re-appear
# def risk_overlap(group):
#     prev = set()
#     vals = []
#     for risks in group['risks']:
#         rs = {r.lower().strip() for r in risks if isinstance(r, str)}
#         vals.append(len(rs & prev) / max(1, len(rs)) if prev else np.nan)
#         prev = rs
#     return pd.Series(vals, index=group.index)
# features['risk_persistence'] = features.groupby('ticker', group_keys=False).apply(risk_overlap)

# # Join returns
# features = features.merge(returns_df, on=['ticker','quarter','call_date'], how='left')
# features.head()

## 6. Train / test split on the first 5–6 calls per ticker

Per the assignment: train on the earliest 5–6 calls per ticker, test on everything after.

In [ ]:
N_TRAIN_PER_TICKER = 5

def split_train_test(df: pd.DataFrame, n_train: int = N_TRAIN_PER_TICKER):
    df = df.sort_values(['ticker','call_date']).copy()
    df['call_idx'] = df.groupby('ticker').cumcount()
    train = df[df.call_idx <  n_train]
    test  = df[df.call_idx >= n_train]
    return train, test

# train, test = split_train_test(features)
# print(f'train: {len(train)}  test: {len(test)}')

## 7. Baseline model + toy backtest

A transparent single-feature baseline: *go long if sentiment > 0, short otherwise, at T+1 close, hold 21 trading days.*
Beat this with your own model.

In [ ]:
HORIZON = 21
RET_COL = f'fwd_excess_{HORIZON}d'

def baseline_signal(df: pd.DataFrame) -> pd.Series:
    return np.sign(df['sentiment'].fillna(0))   # -1 / 0 / +1

def backtest(df: pd.DataFrame, signal: pd.Series) -> dict:
    d = df.assign(signal=signal).dropna(subset=['signal', RET_COL])
    d['pnl'] = d['signal'] * d[RET_COL]
    hit  = (np.sign(d['pnl']) > 0).mean()
    ic   = d[['signal', RET_COL]].corr(method='spearman').iloc[0,1]
    avg  = d.pnl.mean()
    sharpe = avg / (d.pnl.std() + 1e-9) * np.sqrt(252 / HORIZON)
    return {'n': len(d), 'hit_rate': hit, 'rank_IC': ic, 'avg_excess': avg, 'naive_sharpe': sharpe}

# sig = baseline_signal(test)
# print(backtest(test, sig))

# Cumulative PnL plot (uncomment once features are built)
# import matplotlib.pyplot as plt
# d = test.assign(signal=baseline_signal(test)).dropna(subset=['signal', RET_COL]).sort_values('call_date')
# d['cum'] = (d['signal'] * d[RET_COL]).cumsum()
# d.plot(x='call_date', y='cum', title=f'Toy backtest — cumulative excess return ({HORIZON}d holds)')

## 8. Ideas to push past the baseline

- Replace the generic LLM extraction with a **FinBERT** sentiment layer on each speaker block, then an LLM for events only.
- Build a **CEO sentiment** vs. **CFO sentiment** vs. **analyst sentiment** feature triple — analysts often turn negative before the stock does.
- Detect **reactive-vs-proactive** executives: topics raised by analysts in Q&A that were not in prepared remarks.
- Build a cross-sectional long-short: each quarter, long the top-3 tickers by your score, short the bottom-3.
- Try two LLMs (e.g. `gemma3:4b` vs. `qwen3:14b`) on the same prompt and report extraction agreement.
- Add simple price-momentum features (trailing 21d return, distance from 52-week high) alongside your NLP features and see how much the NLP actually contributes.

Good luck. Remember: a clean, honest toy backtest beats a p-hacked 'amazing' one.